# Entrenamiento Clasificador Tipo de Piel → TF.js

**Objetivo:** Entrenar MobileNetV2 en Fitzpatrick17k y exportar a TensorFlow.js para uso en navegador.

**Clases objetivo (5):** `seca`, `grasa`, `mixta`, `sensible`, `normal`

**Mapeo Fitzpatrick I-VI:**
- I → sensible
- II → seca
- III → normal
- IV → mixta
- V → grasa
- VI → grasa

In [ ]:
# Verificar GPU
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Instalar dependencias
!pip install -q tensorflowjs pandas scikit-learn

import os, json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
import tensorflowjs as tfjs

In [ ]:
# Configuración
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS_HEAD = 10
EPOCHS_FINE = 15
LR_HEAD = 1e-3
LR_FINE = 1e-5
DATA_DIR = Path("/content/fitzpatrick17k")
MODEL_DIR = Path("/content/tfjs_model")

CLASS_NAMES = ["seca", "grasa", "mixta", "sensible", "normal"]
NUM_CLASSES = len(CLASS_NAMES)

FITZ_TO_SKIN = {1: "sensible", 2: "seca", 3: "normal", 4: "mixta", 5: "grasa", 6: "grasa"}

In [ ]:
# Descargar dataset
!mkdir -p {DATA_DIR}/images

csv_url = "https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv"
!wget -q -O {DATA_DIR}/fitzpatrick17k.csv {csv_url}

df = pd.read_csv(DATA_DIR / "fitzpatrick17k.csv")
df = df.dropna(subset=['url', 'fitzpatrick'])
df['fitzpatrick'] = df['fitzpatrick'].astype(int)
df['skin_type'] = df['fitzpatrick'].map(FITZ_TO_SKIN)

print(f"Total: {len(df)}")
print(df['skin_type'].value_counts())

# Balancear: 400 por clase (ajustar según tiempo)
df_balanced = df.groupby('skin_type').apply(
    lambda x: x.sample(min(400, len(x)), random_state=42)
).reset_index(drop=True)
print(f"Balanceado: {len(df_balanced)}")
print(df_balanced['skin_type'].value_counts())

In [ ]:
# Descargar imágenes (paralelo para velocidad)
import concurrent.futures, urllib.request

def download_img(row):
    idx, r = row
    path = DATA_DIR / "images" / f"{idx}.jpg"
    if not path.exists():
        try:
            urllib.request.urlretrieve(r['url'], path)
        except:
            pass

with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
    list(executor.map(download_img, df_balanced.iterrows()))

print(f"Imágenes descargadas: {len(list((DATA_DIR/'images').glob('*.jpg')))}")

In [ ]:
# Crear datasets tf.data
def make_dataset(df, training=True):
    paths = [str(DATA_DIR / "images" / f"{idx}.jpg") for idx in df.index]
    labels = [CLASS_NAMES.index(l) for l in df['skin_type']]
    
    def load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = tf.cast(img, tf.float32) / 255.0
        return img, label
    
    def augment(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        img = tf.image.random_saturation(img, 0.8, 1.2)
        img = tf.image.random_hue(img, 0.1)
        return tf.clip_by_value(img, 0, 1), label
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(1000)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_df, val_df = train_test_split(df_balanced, test_size=0.2, 
                                     stratify=df_balanced['skin_type'], random_state=42)
train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

# Class weights
y_train = [CLASS_NAMES.index(l) for l in train_df['skin_type']]
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = {i: w for i, w in enumerate(cw)}

In [ ]:
# Modelo: MobileNetV2 + head
base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = models.Model(inputs, outputs)
model.summary()

In [ ]:
# Fase 1: Head only
model.compile(
    optimizer=optimizers.Adam(LR_HEAD),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_acc')]
)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(patience=2, factor=0.5)
    ]
)

In [ ]:
# Fase 2: Fine-tuning
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(LR_FINE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_acc')]
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(patience=4, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(patience=2, factor=0.5)
    ]
)

In [ ]:
# Evaluar
results = model.evaluate(val_ds, return_dict=True)
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Top-2 Acc: {results['top2_acc']:.4f}")

In [ ]:
# Exportar a TF.js
MODEL_DIR.mkdir(parents=True, exist_ok=True)
tfjs.converters.save_keras_model(model, str(MODEL_DIR))

metadata = {
    "class_names": CLASS_NAMES,
    "img_size": IMG_SIZE,
    "fitz_mapping": FITZ_TO_SKIN,
    "model_type": "MobileNetV2 + custom head",
    "training_samples": int(len(train_df)),
    "val_accuracy": float(results['accuracy'])
}
with open(MODEL_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Modelo exportado en:", MODEL_DIR)
!ls -la {MODEL_DIR}

In [ ]:
# Descargar modelo entrenado
from google.colab import files
import zipfile

zip_path = "/content/tfjs_model.zip"
with zipfile.ZipFile(zip_path, 'w') as z:
    for f in MODEL_DIR.rglob('*'):
        z.write(f, f.relative_to(MODEL_DIR.parent))

files.download(zip_path)
print("Descarga tfjs_model.zip y extrae en tu proyecto")